In [1]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util

import spacy
from rank_bm25 import BM25Okapi

import pathlib, os

import pandas as pd

from tqdm.notebook import tqdm as tqdm_notebook
from tqdm import tqdm

from concurrent.futures import ProcessPoolExecutor,as_completed

import multiprocessing

/home/michele/.local/lib/python3.10/site-packages/beir/datasets/data_loader.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
def data_preparation(dataset:str):
       
       # Download dataset and unzip the dataset
       url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
       out_dir = os.path.join(pathlib.Path(os.path.abspath('')), "datasets")
       data_path = util.download_and_unzip(url, out_dir)
       
       # train/test split
       d_train,q_train,qr_train=GenericDataLoader(data_path).load(split="train")
       _,q_test,qr_test=GenericDataLoader(data_path).load(split="test")
       
       # to pandas dataframe
       train={"documents":pd.DataFrame.from_dict(d_train,orient="index").drop(["title"], axis=1),
              "queries":pd.DataFrame.from_dict(q_train,orient="index",columns=["text"]),
              "real":qr_train}
       
       test={"queries":pd.DataFrame.from_dict(q_test,orient="index",columns=["text"]),
              "real":qr_test}
              
              
       def data_tokenizer(train:dict[pd.DataFrame],test:dict[pd.DataFrame]):
              nlp = spacy.load("en_core_web_sm", disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer", "ner"])
              tqdm_notebook.pandas()
              
              def tokenizer(text):
                     #return " ".join(token.text for token in nlp(text) if not token.is_stop and not token.is_punct)
                     return [token.text for token in nlp(text) if not token.is_stop and not token.is_punct]
              
              train["documents"]["text"]=train["documents"]["text"].progress_apply(tokenizer)
              train["queries"]["text"]=train["queries"]["text"].progress_apply(tokenizer)
              
              #test["documents"]["text"]=test["documents"]["text"].progress_apply(tokenizer)
              test["queries"]["text"]=test["queries"]["text"].progress_apply(tokenizer)
       
       
       #print(train["documents"].equals(test["documents"]))
       #print(train["queries"].equals(test["queries"]))
       
       #data tokenization using spacy
       data_tokenizer(train,test)
       
       return train,test

train,test=data_preparation("fiqa")

  0%|          | 0/57638 [00:00<?, ?it/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

  0%|          | 0/5500 [00:00<?, ?it/s]

  0%|          | 0/648 [00:00<?, ?it/s]

In [3]:
bar_queue = multiprocessing.Queue()

def BM25(bm25,start,stop):
    
    scores_per_query={}

    for index,row in train["queries"].iloc[start:stop].iterrows():
        
        scores=bm25.get_scores(row["text"])
        
        scores_per_query[index]=scores
        
        bar_queue.put_nowait(1)

    return scores_per_query

def BM25_Multiprocessing(n_jobs=8):
    
    scores_per_query_total={}
    
    def bar_handler(q):
        bar=tqdm(desc="Total", total=len(train["queries"]),bar_format="{l_bar}{bar}|{n_fmt}/{total_fmt} [{elapsed}]")
        
        while True:
            how_much=q.get()
            bar.update(how_much)
            bar.refresh()

    bar_process = multiprocessing.Process(target=bar_handler, args=(bar_queue, ), daemon=True)
    bar_process.start()

    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        
        bm25 = BM25Okapi(train["documents"]["text"].tolist())
        
        length=int(len(train["queries"])/n_jobs)
        futures=[]
            
        for start in range(0,len(train["queries"]),length):
            
            futures.append( executor.submit(BM25,bm25,start,start+length) )
        
        for future in as_completed(futures):
            result=future.result()
            scores_per_query_total.update(result)

    bar_process.terminate()
    
    return scores_per_query_total

scores_per_query=BM25_Multiprocessing(8)

del bar_queue

Total:   3%|▎         |142/5500 [00:10]

In [ ]:
scores_per_query

{'2742': array([0.        , 2.57400826, 4.38023321, ..., 0.        , 0.        ,
        0.        ]),
 '2743': array([0.        , 0.        , 3.76916795, ..., 0.        , 2.58514043,
        0.        ]),
 '2745': array([0., 0., 0., ..., 0., 0., 0.]),
 '2748': array([0., 0., 0., ..., 0., 0., 0.]),
 '2751': array([0., 0., 0., ..., 0., 0., 0.]),
 '2752': array([0., 0., 0., ..., 0., 0., 0.]),
 '2753': array([0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        6.19846939]),
 '2754': array([0., 0., 0., ..., 0., 0., 0.]),
 '2756': array([0.        , 3.12841425, 0.        , ..., 0.        , 0.        ,
        0.        ]),
 '2757': array([0., 0., 0., ..., 0., 0., 0.]),
 '2758': array([0., 0., 0., ..., 0., 0., 0.]),
 '2763': array([0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        2.95589135]),
 '2764': array([0., 0., 0., ..., 0., 0., 0.]),
 '2765': array([0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        5.9117827]),
 '2768': arr

In [ ]:
train["documents"]["text"].tolist()

[['saying',
  'like',
  'idea',
  'job',
  'training',
  'expect',
  'company',
  'Training',
  'workers',
  'job',
  'building',
  'software',
  'educational',
  'systems',
  'U.S.',
  'students',
  'worry',
  'little',
  'getting',
  'marketable',
  'skills',
  'exchange',
  'massive',
  'investment',
  'education',
  'getting',
  'thousands',
  'student',
  'debt',
  'complaining',
  'qualified'],
 ['preventing',
  'false',
  'ratings',
  'additional',
  'scrutiny',
  'market',
  'investors',
  'newer',
  'controls',
  'place',
  'prevent',
  'institutions',
  'DFA',
  'banks',
  'longer',
  'solely',
  'rely',
  'credit',
  'ratings',
  'diligence',
  'buy',
  'financial',
  'instrument',
  'plus',
  'intent',
  'financial',
  'institutions',
  'leg',
  'work',
  'maybe',
  'figure',
  'certain',
  'CDO',
  'garbage',
  ' ',
  'Edit',
  'lead'],
 ['use',
  'health',
  'FSA',
  'individual',
  'health',
  'insurance',
  'premiums',
  ' ',
  'FSA',
  'plan',
  'sponsors',
  'limit',
